Initializing GEE

In [2]:
import ee
import earthaccess
import geemap

ee.Authenticate()
ee.Initialize(project='conordoremus-project')

In [ ]:
# importing packages
import rioxarray
import re

Importing class and methods from site selector

In [3]:
import sys
sys.path.insert(0, "/Users/conordoremus/GitHub/tpae/src")

from absolute_effectiveness.site_selector import SiteSelector
selector = SiteSelector()


# Selecting Sites

Example PAS <br>

* Shasta Big Springs Ranch - 11116292
* Pinecone Burke - 101664
* Chena River - 68399

In [4]:
test_sites = selector.get_test_sites()

tergola = selector.get_site_geom(test_sites, 555786096)
pinecone_burke = selector.get_site_geom(test_sites, 101664)
chena_river = selector.get_site_geom(test_sites, 68399)

def get_bounds(geom):
    coords = geom.bounds().coordinates().getInfo()[0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

tergola_bounds = get_bounds(tergola)  
pinecone_bounds = get_bounds(pinecone_burke)
chena_bounds = get_bounds(chena_river)


Accessing DIST ALERT data from NASA earthdata

In [5]:
auth = earthaccess.login(strategy="netrc")
print(auth.authenticated)

True


In [4]:
results = earthaccess.search_data(
    short_name="OPERA_L3_DIST-ALERT-HLS_V1",
    bounding_box=(*tergola_bounds,),
    temporal=("2023-01-01", "2024-01-01")
)
print(f"Found {len(results)} granules")


Found 149 granules


In [6]:
g = results[0]

# Human-readable summary
print(g)

# All the S3/HTTPS links for that granule's files
print(g.data_links())

# Check if this granule lives in the cloud
print(g.cloud_hosted)  # True for OPERA data

Collection: {'ShortName': 'OPERA_L3_DIST-ALERT-HLS_V1', 'Version': '1'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': 90.12117667, 'Latitude': 26.99956652}, {'Longitude': 90.14924457, 'Latitude': 27.98934636}, {'Longitude': 89.08001682, 'Latitude': 28.00942221}, {'Longitude': 89.02987038, 'Latitude': 27.80712738}, {'Longitude': 89.01552348, 'Latitude': 27.01974136}, {'Longitude': 90.12117667, 'Latitude': 26.99956652}]}}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2023-01-01T04:30:15.426402Z', 'EndingDateTime': '2023-01-01T04:30:15.426402Z'}}
Size(MB): 0
Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ALERT-HLS_V1/OPERA_L3_DIST-ALERT-HLS_T45RYL_20230101T043015Z_20231222T014957Z_L9_30_v1/OPERA_L3_DIST-ALERT-HLS_T45RYL_20230101T043015Z_20231222T014957Z_L9_30_v1_VEG-DIST-STATUS.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ALERT-HLS

In [8]:
files = earthaccess.open([results[0]])

QUEUEING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
# Pick a specific layer — e.g. vegetation disturbance status
target_layer = "VEG-DIST-STATUS"

layer_files = [f for f in files if target_layer in f.path]

# Open one tile
ds = rioxarray.open_rasterio(layer_files[0])
print(ds)

<xarray.DataArray (band: 1, y: 3660, x: 3660)> Size: 13MB
[13395600 values with dtype=uint8]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 29kB 3.1e+06 3.1e+06 3.1e+06 ... 2.99e+06 2.99e+06
  * x            (x) float64 29kB 7e+05 7e+05 7e+05 ... 8.097e+05 8.097e+05
    spatial_ref  int64 8B 0
Attributes: (12/13)
    flag_meanings:             no_disturbance,firstdetection_<50%,provisional...
    flag_values:               0,1,2,3,4,5,6,7,8,255
    Input_DIST-ALERT_granule:  first
    Percent_Updated:           21.43
    Units:                     unitless
    Update_Date:               2023-01-01T04:30:15.4264029Z
    ...                        ...
    Valid_min:                 0
    AREA_OR_POINT:             Area
    _FillValue:                255
    scale_factor:              1.0
    add_offset:                0.0
    long_name:                 Vegetation_disturbance_status


# Using Earth Engine - Recent Composite

In [6]:
# importing recent mosaic with EE

folder = "projects/glad/HLSDIST/current"
VEGDISTSTATUS = ee.ImageCollection(folder+"/VEG-DIST-STATUS").mosaic()
VEGDISTDATE = ee.ImageCollection(folder+"/VEG-DIST-DATE").mosaic()
WDPA = ee.FeatureCollection("WCMC/WDPA/current/polygons")

In [ ]:
# datepal = ["#22318DFF","#2355A4FF","#1F80B8FF","#31A6C2FF","#5DC0C0FF","#97D6B9FF","#CFECB3FF","#EFF9B5FF","#FFFFD9FF"]
# durpal = ["#231151FF","#410F75FF","#5F187FFF","#7B2382FF","#982D80FF","#B63679FF","#D3436EFF","#EB5760FF","#F8765CFF","#FD9969FF","#FEBA80FF","#FDDC9EFF","#FCFDBFFF"]
# anompal = ["444444","#FEE187FF","#FEC965FF","#FEAB49FF","#FD8D3CFF","#FC5B2EFF","#ED2F22FF","#D41020FF","#B10026FF","#800026FF"]
# confpal = ['000000',"#FFFFCCFF","#FFEFA5FF","#FEDD7FFF","#FEBF5AFF","#FD9D43FF","#FD7134FF","#F43D25FF","#DB141EFF","#B60026FF","#800026FF"]
# genanompal = ["#440154FF","#481B6DFF","#46337EFF","#3F4889FF","#365C8DFF","#2E6E8EFF","#277F8EFF","#21908CFF",'#1FA187FF','#2DB27DFF','#4AC16DFF','#71CF57FF','#9FDA3AFF','#CFE11CFF','#FDE725FF']
# indpal = ["#F7FCF5FF","#E5F5E0FF","#C7E9C0FF","#A1D99BFF","#74C476FF","#41AB5DFF","#238B45FF","#006D2CFF","#00441BFF"]
# statuspal = ["121212","005555","897f4e","dee043","008888","e48727","e01b07","777777","dddddd"]

# today = ee.Date(new Date())
# todayJ = today.difference("2020-12-31",'day').int().getInfo()
# prevyeardate = today.advance(-1,"year")
# prevyeardateJ = prevyeardate.difference("2020-12-31",'day').int().getInfo()

In [12]:
pinecone_burke_DIST = VEGDISTSTATUS.clip(pinecone_burke)
tergola_DIST = VEGDISTSTATUS.clip(tergola)
chena_DIST = VEGDISTSTATUS.clip(chena_river)

statuspal = ["121212","005555","897f4e","dee043","008888","e48727","e01b07","777777","dddddd"]


In [14]:
def get_buffer_geom(site_geom_ee, buffer_m=10000):
    """Returns donut: buffer zone excluding PA interior."""
    outer = site_geom_ee.buffer(buffer_m)
    return outer.difference(site_geom_ee)


In [17]:
tergola_buffer = get_buffer_geom(tergola)
tergola_buffer_DIST = VEGDISTSTATUS.clip(tergola_buffer)

In [21]:
dist_mask = VEGDISTSTATUS.gt(0).selfMask()
tergola_DIST_pixel = dist_mask.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=tergola,
    scale=30,
    maxPixels=1e9
) 

tergola_total_pixel = VEGDISTSTATUS.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=tergola, 
    scale=30,
    maxPixels=1e9
)

print(tergola_DIST_pixel.getInfo())
print(tergola_total_pixel.getInfo())

tergola_dist_rate_inside = tergola_DIST_pixel.getInfo()['b1'] / tergola_total_pixel.getInfo()['b1']
print(f"Tergola Disturbance Rate Inside PA: {tergola_dist_rate_inside}")

{'b1': 674}
{'b1': 39615}
Tergola Disturbance Rate Inside PA: 0.017013757415120536


In [22]:

tergola_buffer_DIST_pixel = dist_mask.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=tergola_buffer,
    scale=30,
    maxPixels=1e9
) 

tergola_buffer_total_pixel = VEGDISTSTATUS.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=tergola_buffer, 
    scale=30,
    maxPixels=1e9
)

print(tergola_buffer_DIST_pixel.getInfo())
print(tergola_buffer_total_pixel.getInfo())

tergola_buffer_dist_rate_inside = tergola_buffer_DIST_pixel.getInfo()['b1'] / tergola_buffer_total_pixel.getInfo()['b1']
print(f"Tergola Buffer Disturbance Rate Inside PA: {tergola_buffer_dist_rate_inside}")

{'b1': 13586}
{'b1': 675174}
Tergola Buffer Disturbance Rate Inside PA: 0.02012222034616262


In [23]:
tergola_effectiveness = 1 - (tergola_dist_rate_inside / tergola_buffer_dist_rate_inside)
print(f"Tergola Effectiveness: {tergola_effectiveness}")

Tergola Effectiveness: 0.15447912196403701


In [18]:
Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")

Map.addLayer(tergola, {}, "Tergola Site Boundary")
Map.addLayer(tergola_DIST, {'min':0,'max':8,'palette':statuspal}, "DIST Alert - Tergola")
Map.addLayer(tergola_buffer, {'color': 'FF0000'}, "Tergola 10km Buffer")
Map.addLayer(tergola_buffer_DIST, {'min':0,'max':8,'palette':statuspal}, "DIST Alert - Tergola Buffer")

Map.centerObject(tergola_DIST, 10)
Map

Map(center=[27.1874534353081, 89.20468885868115], controls=(WidgetControl(options=['position', 'transparent_bg…